In [1]:
import numpy as np
import pandas as pd
print("numpy_version:",np.__version__ )
print ("Pandas_Version:", pd.__version__) 

numpy_version: 2.3.5
Pandas_Version: 2.3.3


In [2]:
## Load the dataset and cleaning process ## 
import numpy as np 
import pandas as pd
df = pd.read_csv("emi_prediction_dataset.csv")
print(df.shape)
print(df.columns.tolist())   # check the real column names
df.info()
print(df.isna().sum())
print("Duplicates:", df.duplicated().sum())
print(df["emi_eligibility"].value_counts())

C:\Users\hp\AppData\Local\Temp\ipykernel_3152\3069326227.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("emi_prediction_dataset.csv")


(404800, 27)
['age', 'gender', 'marital_status', 'education', 'monthly_salary', 'employment_type', 'years_of_employment', 'company_type', 'house_type', 'monthly_rent', 'family_size', 'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities', 'other_monthly_expenses', 'existing_loans', 'current_emi_amount', 'credit_score', 'bank_balance', 'emergency_fund', 'emi_scenario', 'requested_amount', 'requested_tenure', 'emi_eligibility', 'max_monthly_emi']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404800 entries, 0 to 404799
Data columns (total 27 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   age                     404800 non-null  object 
 1   gender                  404800 non-null  object 
 2   marital_status          404800 non-null  object 
 3   education               402396 non-null  object 
 4   monthly_salary          404800 non-null  object 
 5   employment_type         40

In [3]:
df.head()

,age,gender,marital_status,education,monthly_salary,employment_type,years_of_employment,company_type,house_type,monthly_rent,...,existing_loans,current_emi_amount,credit_score,bank_balance,emergency_fund,emi_scenario,requested_amount,requested_tenure,emi_eligibility,max_monthly_emi
0,38.0,Female,Married,Professional,82600.0,Private,0.9,Mid-size,Rented,20000.0,...,Yes,23700.0,660.0,303200.0,70200.0,Personal Loan EMI,850000.0,15,Not_Eligible,500.0
1,38.0,Female,Married,Graduate,21500.0,Private,7.0,MNC,Family,0.0,...,Yes,4100.0,714.0,92500.0,26900.0,E-commerce Shopping EMI,128000.0,19,Not_Eligible,700.0
2,38.0,Male,Married,Professional,86100.0,Private,5.8,Startup,Own,0.0,...,No,0.0,650.0,672100.0,324200.0,Education EMI,306000.0,16,Eligible,27775.0
3,58.0,Female,Married,High School,66800.0,Private,2.2,Mid-size,Own,0.0,...,No,0.0,685.0,440900.0,178100.0,Vehicle EMI,304000.0,83,Eligible,16170.0
4,48.0,Female,Married,Professional,57300.0,Private,3.4,Mid-size,Family,0.0,...,No,0.0,770.0,97300.0,28200.0,Home Appliances EMI,252000.0,7,Not_Eligible,500.0


In [3]:
## fill medien if value is missed in a column ,working on categorial data  ##

# 1. Remove duplicate rows
df = df.drop_duplicates().copy()

# 2. Convert financial columns into numeric data types
numeric_cols_to_convert = [
    "monthly_rent", "school_fees", "college_fees",
    "travel_expenses", "groceries_utilities",
    "other_monthly_expenses", "current_emi_amount",
    "monthly_salary", "requested_amount", "requested_tenure"
]

for c in numeric_cols_to_convert:
    if c in df.columns:
        df[c] = pd.to_numeric(
            df[c].astype("string")
                 .str.replace(r"[₹,\s]", "", regex=True),
            errors="coerce"
        )

# 3. Fill missing numerical values using median   (EDA )
num_cols = df.select_dtypes(include="number").columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# 4. Fill missing categorical values using mode
cat_cols = df.select_dtypes(include=["object", "string", "category"]).columns

for c in cat_cols:
    mode_values = df[c].mode()
    if not mode_values.empty:
        df[c] = df[c].fillna(mode_values.iloc[0])

# 5. Verify cleaning
print("Remaining nulls:", df.isna().sum().sum())
print("Remaining duplicates:", df.duplicated().sum())
print("\nData types:")
print(df.dtypes)


Remaining nulls: 0
Remaining duplicates: 0

Data types:
age                        object
gender                     object
marital_status             object
education                  object
monthly_salary            Float64
employment_type            object
years_of_employment       float64
company_type               object
house_type                 object
monthly_rent              Float64
family_size                 int64
dependents                  int64
school_fees               Float64
college_fees              Float64
travel_expenses           Float64
groceries_utilities       Float64
other_monthly_expenses    Float64
existing_loans             object
current_emi_amount        Float64
credit_score              float64
bank_balance               object
emergency_fund            float64
emi_scenario               object
requested_amount          Float64
requested_tenure            Int64
emi_eligibility            object
max_monthly_emi           float64
dtype: object


In [5]:
## Adavance engineering ###
eps = 1  # avoids division by zero

expense_cols = ["monthly_rent", "school_fees", "college_fees",
                "travel_expenses", "groceries_utilities", "other_monthly_expenses"]
df["total_monthly_expenses"] = df[expense_cols].sum(axis=1)
df["debt_to_income"] = df["current_emi_amount"] / (df["monthly_salary"] + eps)
df["expense_to_income"] = df["total_monthly_expenses"] / (df["monthly_salary"] + eps)
df["affordability_ratio"] = (df["monthly_salary"] - df["total_monthly_expenses"]
                             - df["current_emi_amount"]) / (df["monthly_salary"] + eps)
df["loan_to_salary"] = df["requested_amount"] / (df["monthly_salary"] + eps)
df["monthly_loan_burden"] = df["requested_amount"] / (df["requested_tenure"] + eps)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Categorical columns to one-hot encode (only columns with few unique values)
cat_cols = ["gender", "marital_status", "education", "employment_type",
            "company_type", "house_type", "emi_scenario", "existing_loans"]
cat_cols = [c for c in cat_cols if c in df.columns]

# 2. Numeric feature columns (targets and text columns excluded)
drop_cols = ["emi_eligibility", "max_monthly_emi"]
num_cols = [c for c in df.columns
            if c not in drop_cols + cat_cols and pd.api.types.is_numeric_dtype(df[c])]

# 3. Build X: numeric columns + one-hot columns for the chosen categories only
X = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, drop_first=True, dtype=int)
print("X shape:", X.shape)
assert X.shape[1] < 100, f"Too many columns: {X.shape[1]}. Check cat_cols."

# 4. Targets
le = LabelEncoder()
y_clf = le.fit_transform(df["emi_eligibility"])
y_reg = df["max_monthly_emi"].values

# 5. Split: 70% train, 15% validation, 15% test (stratified on the class)
X_temp, X_test, yc_temp, yc_test, yr_temp, yr_test = train_test_split(
    X, y_clf, y_reg, test_size=0.15, random_state=42, stratify=y_clf)
X_train, X_val, yc_train, yc_val, yr_train, yr_val = train_test_split(
    X_temp, yc_temp, yr_temp, test_size=0.1765, random_state=42, stratify=yc_temp)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

X shape: (404800, 39)
Train: (283349, 39) Val: (60731, 39) Test: (60720, 39)


In [5]:
import mlflow
mlflow.set_tracking_uri("sqlite:///C:/Users/hp/Desktop/intership project 1/mlflow.db")   # SQLite backend supports the model registry
mlflow.set_experiment("emi_classification")

<Experiment: artifact_location='file:C:/Users/hp/Desktop/intership project 1/mlruns/1', creation_time=1792594929032, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792594929032, lifecycle_stage='active', name='emi_classification', tags={}, trace_location=None, workspace='default'>

In [13]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

clf_models = {
    "logistic_regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "xgboost": XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6, eval_metric="mlogloss"),
}

for name, model in clf_models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train, yc_train)

        pred = model.predict(X_val)
        proba = model.predict_proba(X_val)
        metrics = {
            "accuracy": accuracy_score(yc_val, pred),
            "precision_macro": precision_score(yc_val, pred, average="macro"),
            "recall_macro": recall_score(yc_val, pred, average="macro"),
            "f1_macro": f1_score(yc_val, pred, average="macro"),
            "roc_auc_ovr": roc_auc_score(yc_val, proba, multi_class="ovr"),
        }
        mlflow.log_param("model_name", name)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, "model", serialization_format="cloudpickle")
        print(name, {k: round(v, 4) for k, v in metrics.items()})

2026/10/21 21:47:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/21 21:47:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


logistic_regression {'accuracy': 0.89, 'precision_macro': 0.5873, 'recall_macro': 0.5841, 'f1_macro': 0.5741, 'roc_auc_ovr': 0.9222}


2026/10/21 21:50:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/21 21:50:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


random_forest {'accuracy': 0.9209, 'precision_macro': 0.9312, 'recall_macro': 0.6176, 'f1_macro': 0.6195, 'roc_auc_ovr': 0.9529}


2026/10/21 21:51:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/21 21:51:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


xgboost {'accuracy': 0.9484, 'precision_macro': 0.8277, 'recall_macro': 0.6742, 'f1_macro': 0.6774, 'roc_auc_ovr': 0.9804}


In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

mlflow.set_experiment("emi_regression")

reg_models = {
    "linear_regression": make_pipeline(StandardScaler(), LinearRegression()),
    "random_forest_reg": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "xgboost_reg": XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6),
}

for name, model in reg_models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train, yr_train)
        pred = model.predict(X_val)
        metrics = {
            "rmse": np.sqrt(mean_squared_error(yr_val, pred)),
            "mae": mean_absolute_error(yr_val, pred),
            "r2": r2_score(yr_val, pred),
            "mape": mean_absolute_percentage_error(yr_val, pred),
        }
        mlflow.log_param("model_name", name)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, "model", serialization_format="cloudpickle")
        print(name, {k: round(v, 4) for k, v in metrics.items()})

2026/10/21 22:12:54 INFO mlflow.tracking.fluent: Experiment with name 'emi_regression' does not exist. Creating a new experiment.
2026/10/21 22:12:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/21 22:12:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


linear_regression {'rmse': np.float64(4243.2577), 'mae': 3029.5858, 'r2': 0.7025, 'mape': 1.9382}


2026/10/21 22:28:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/21 22:28:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


random_forest_reg {'rmse': np.float64(1500.6493), 'mae': 854.6179, 'r2': 0.9628, 'mape': 0.2404}


2026/10/21 22:31:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/21 22:31:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


xgboost_reg {'rmse': np.float64(1161.7559), 'mae': 726.8059, 'r2': 0.9777, 'mape': 0.407}


In [15]:
import os
import joblib
import mlflow

os.makedirs("../models", exist_ok=True)

# 1. Register the best regression run (lowest RMSE)
best_reg_run = mlflow.search_runs(experiment_names=["emi_regression"],
                                  order_by=["metrics.rmse ASC"]).iloc[0]
print("Best regressor run:", best_reg_run["tags.mlflow.runName"],
      "RMSE =", round(best_reg_run["metrics.rmse"], 2))
mlflow.register_model(f"runs:/{best_reg_run.run_id}/model", "emi_regressor")

# 2. Save the fitted models and metadata for the Streamlit app
joblib.dump(clf_models["xgboost"], "../models/emi_classifier.joblib")
joblib.dump(reg_models["xgboost_reg"], "../models/emi_regressor.joblib")
joblib.dump(le, "../models/label_encoder.joblib")
joblib.dump(list(X.columns), "../models/feature_columns.joblib")

print("Saved: emi_classifier, emi_regressor, label_encoder, feature_columns")

Best regressor run: xgboost_reg RMSE = 1161.76


Successfully registered model 'emi_regressor'.
2026/10/21 22:34:38 WARNING mlflow.tracking._model_registry.fluent: Run with id 566376591b4a4e7fb0cac72c298ba66d has no artifacts at artifact path 'model', registering model based on models:/m-ea385ede946f47edaf08bcd6c5ef5e73 instead
Created version '1' of model 'emi_regressor'.


Saved: emi_classifier, emi_regressor, label_encoder, feature_columns


In [29]:
import streamlit as st, joblib, pandas as pd

clf = joblib.load("models/emi_classifier.joblib")
reg = joblib.load("models/emi_regressor.joblib")
cols = joblib.load("models/feature_columns.joblib")

st.title("EMIPredict AI")
salary = st.number_input("Monthly salary (INR)", 15000, 200000, 50000)
# ...add the other inputs, build a one-row DataFrame with the same engineered features,
# align it to `cols`, then call clf.predict and reg.predict

2026-10-21 23:08:56.134 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-21 23:08:56.273 
  command:

    streamlit run C:\ProgramData\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-10-21 23:08:56.274 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-21 23:08:56.275 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-21 23:08:56.279 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-21 23:08:56.281 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-21 23:08:56.282 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-21 23:08:56.287 Thread 'MainThread': mi

In [30]:
i = X_test.index[0]          # or any index you like
row = X_test.loc[[i]]
print("Notebook label:", le.inverse_transform(clf_models["xgboost"].predict(row))[0])
print("Notebook max EMI:", round(reg_models["xgboost_reg"].predict(row)[0], 2))
print("Actual label:", df.loc[i, "emi_eligibility"])
print("Actual max EMI:", df.loc[i, "max_monthly_emi"])
print(df.loc[i, ["monthly_salary", "monthly_rent", "groceries_utilities", "travel_expenses",
                 "other_monthly_expenses", "current_emi_amount", "requested_amount",
                 "requested_tenure", "credit_score", "bank_balance", "emergency_fund"]])

Notebook label: Not_Eligible
Notebook max EMI: 342.45
Actual label: Not_Eligible
Actual max EMI: 1083.6
monthly_salary            428709.0
monthly_rent                   0.0
groceries_utilities         6200.0
travel_expenses             2900.0
other_monthly_expenses      2400.0
current_emi_amount          6500.0
requested_amount           65000.0
requested_tenure                36
credit_score                 694.0
bank_balance               30500.0
emergency_fund             18300.0
Name: 297497, dtype: object
